# SQL Queries — Piedmont Triad Real Estate Investment Analysis

This notebook contains 12 SQL queries that complement the main analysis in 
`DAPS_Final_EDA.ipynb`. Queries are organized in three thematic groups:

- **Market structure** (Q1, Q2, Q11, Q12) — city-level inventory, price 
  distributions, property type composition
- **Demographic enrichment and value identification** (Q3, Q4, Q5, Q6, Q7, Q8) — 
  joins with Census/market data, within-zip rankings, statistical undervaluation
- **Strategic patterns** (Q9, Q10) — multi-stage CTEs for rental investment 
  profiles and cross-cutting demographic filters

All queries operate on a SQLite in-memory database loaded from the same CSV 
files used by the main notebook. Each query is commented with its purpose, 
SQL technique, and analytical relevance. Findings are integrated into the 
main notebook's EDA narrative and conclusion.

## Setup

In [54]:
import pandas as pd
import sqlite3

# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")

# Load each CSV and write it as a SQL table
df_listings = pd.read_csv("listings_clean.csv", dtype={"zipCode": str})
df_listings.to_sql("listings", conn, index=False, if_exists="replace")

df_census = pd.read_csv("census_data_raw.csv", dtype={"zipCode": str})
df_census.to_sql("census", conn, index=False, if_exists="replace")

df_market = pd.read_csv("market_data_raw.csv", dtype={"zipCode": str})

# Drop the nested columns — SQLite can't query inside lists/dicts
df_market = df_market.drop(columns=["dataByPropertyType", "dataByBedrooms", "history"], errors="ignore")
df_market.to_sql("market", conn, index=False, if_exists="replace")

df_rates = pd.read_csv("mortgage_rates_raw.csv", parse_dates=["date"])
df_rates.to_sql("mortgage_rates", conn, index=False, if_exists="replace")

# Quick verification — list all tables and their row counts
verify_query = """
SELECT name, (SELECT COUNT(*) FROM listings) AS listings_rows
FROM sqlite_master WHERE type='table'
"""

for table in ["listings", "census", "market", "mortgage_rates"]:
    count = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", conn)["n"][0]
    print(f"  {table}: {count} rows")

print("\nDatabase ready.")

  listings: 952 rows
  census: 24 rows
  market: 24 rows
  mortgage_rates: 173 rows

Database ready.


## Query 1 — Listings Inventory and Price Statistics by City

**Purpose:** Establish a baseline view of the Piedmont Triad market by summarizing 
listing inventory and price distribution at the city level. This is the first 
question any investor would ask: "What's available and at what price points 
across the three markets I'm considering?"

In [55]:
query_1 = """
SELECT
    city,
    COUNT(*)                    AS listing_count,
    ROUND(AVG(price), 0)        AS avg_price,
    ROUND(MIN(price), 0)        AS min_price,
    ROUND(MAX(price), 0)        AS max_price,
    ROUND(AVG(squareFootage), 0) AS avg_sqft,
    ROUND(AVG(price / squareFootage), 2) AS avg_price_per_sqft
FROM listings
WHERE squareFootage IS NOT NULL
  AND squareFootage > 0
GROUP BY city
ORDER BY listing_count DESC;
"""

pd.read_sql(query_1, conn)

,city,listing_count,avg_price,min_price,max_price,avg_sqft,avg_price_per_sqft
0,Greensboro,695,354965.0,83000.0,998000.0,1921.0,185.12
1,High Point,257,300771.0,40000.0,999000.0,1816.0,167.25


## Query 2 — Property Type Composition by City (with Subtotals)

**Purpose:** Break down each city's inventory by property type to understand 
market composition. Are single-family homes the dominant product, or is there 
meaningful condo/townhouse inventory? The query produces detail rows per 
(city, property type), city-level subtotals, and a grand total — the same 
output that ROLLUP or GROUPING SETS would produce, implemented via UNION ALL 
because the SQLite distribution in this environment does not support those 
syntaxes.



In [60]:
query_2 = """
SELECT *
FROM (
    -- Detail rows: one per (city, propertyType) combination
    SELECT
        city                                  AS city,
        propertyType                          AS property_type,
        COUNT(*)                              AS listing_count,
        ROUND(AVG(price), 0)                  AS avg_price,
        ROUND(AVG(squareFootage), 0)          AS avg_sqft
    FROM listings
    WHERE squareFootage IS NOT NULL AND squareFootage > 0
    GROUP BY city, propertyType

    UNION ALL

    -- Subtotal rows: one per city across all property types
    SELECT
        city                                  AS city,
        'ALL TYPES'                           AS property_type,
        COUNT(*)                              AS listing_count,
        ROUND(AVG(price), 0)                  AS avg_price,
        ROUND(AVG(squareFootage), 0)          AS avg_sqft
    FROM listings
    WHERE squareFootage IS NOT NULL AND squareFootage > 0
    GROUP BY city

    UNION ALL

    -- Grand total row across all cities and property types
    SELECT
        'ALL CITIES'                          AS city,
        'ALL TYPES'                           AS property_type,
        COUNT(*)                              AS listing_count,
        ROUND(AVG(price), 0)                  AS avg_price,
        ROUND(AVG(squareFootage), 0)          AS avg_sqft
    FROM listings
    WHERE squareFootage IS NOT NULL AND squareFootage > 0
) AS combined
ORDER BY
    CASE WHEN city = 'ALL CITIES' THEN 1 ELSE 0 END,
    city,
    CASE WHEN property_type = 'ALL TYPES' THEN 1 ELSE 0 END,
    listing_count DESC;
"""

pd.read_sql(query_2, conn)

,city,property_type,listing_count,avg_price,avg_sqft
0,Greensboro,Single Family,463,391149.0,2105.0
1,Greensboro,Townhouse,145,328017.0,1765.0
2,Greensboro,Condo,84,204582.0,1177.0
3,Greensboro,Manufactured,2,219900.0,1407.0
4,Greensboro,Multi-Family,1,412000.0,2563.0
5,Greensboro,ALL TYPES,695,354965.0,1921.0
6,High Point,Single Family,211,311526.0,1879.0
7,High Point,Townhouse,31,301270.0,1665.0
8,High Point,Condo,13,149608.0,1115.0
9,High Point,Land,1,70000.0,3260.0


## Query 3 — Listings Joined with Census Demographics

**Purpose:** Enrich each listing with neighborhood demographics by joining on 
zip code. A house's "investability" depends not just on its own attributes 
(price, size, condition) but on the demographic context around it — median 
household income (a proxy for ability to pay) and renter ratio (a proxy for 
rental demand). This join produces the foundational enriched dataset that 
subsequent queries build on.

In [62]:
query_3 = """
SELECT
    l.formattedAddress,
    l.city,
    l.zipCode,
    l.propertyType,
    l.price,
    l.bedrooms,
    l.bathrooms,
    l.squareFootage,
    c.median_household_income,
    c.renter_ratio,
    c.total_population
FROM listings AS l
LEFT JOIN census AS c
    ON l.zipCode = c.zipCode
WHERE l.squareFootage IS NOT NULL
  AND l.squareFootage > 0
ORDER BY c.renter_ratio DESC, l.price ASC
LIMIT 10;
"""

pd.read_sql(query_3, conn)

,formattedAddress,city,zipCode,propertyType,price,bedrooms,bathrooms,squareFootage,median_household_income,renter_ratio,total_population
0,"5661 Hornaday Rd, Unit H, Greensboro, NC 27409",Greensboro,27409,Condo,114000,2.0,1.0,788.0,57862,0.59609,16927
1,"5715 Bramblegate Rd, Unit F, Greensboro, NC 27409",Greensboro,27409,Condo,129900,2.0,2.5,1152.0,57862,0.59609,16927
2,"5515 Hornaday Rd, Unit D, Greensboro, NC 27409",Greensboro,27409,Condo,133000,2.0,1.0,912.0,57862,0.59609,16927
3,"5737 Bramblegate Rd, Unit F, Greensboro, NC 27409",Greensboro,27409,Condo,139999,2.0,2.5,1070.0,57862,0.59609,16927
4,"812 Ashebrook Dr, Apt D, Greensboro, NC 27409",Greensboro,27409,Condo,149900,2.0,2.0,1035.0,57862,0.59609,16927
5,"15 Meadowood Glen Way, Apt C, Greensboro, NC 2...",Greensboro,27409,Condo,155000,2.0,2.0,1102.0,57862,0.59609,16927
6,"5723 Bramblegate Rd, Unit A, Greensboro, NC 27409",Greensboro,27409,Condo,162000,3.0,2.5,1344.0,57862,0.59609,16927
7,"802 Folly Ct, Unit G, Greensboro, NC 27409",Greensboro,27409,Condo,162500,2.0,2.0,1046.0,57862,0.59609,16927
8,"5735 Bramblegate Rd, Unit B, Greensboro, NC 27409",Greensboro,27409,Condo,165000,2.0,2.5,1086.0,57862,0.59609,16927
9,"4290 Cedarcroft Ct, Apt 2A, Greensboro, NC 27409",Greensboro,27409,Condo,169900,2.0,2.0,1161.0,57862,0.59609,16927


## Query 4 — Listings Compared to Zip-Level Market Median

**Purpose:** For each listing, compute how much it deviates from its zip code's 
market median price. A listing priced 30% below its zip's median may be an 
underpriced opportunity (or a fixer-upper). This query lays the groundwork for 
identifying value buys by surfacing the listings furthest below their local 
median.

In [63]:
query_4 = """
SELECT
    l.formattedAddress,
    l.city,
    l.zipCode,
    l.propertyType,
    l.price                                            AS listing_price,
    m.medianPrice                                      AS zip_median_price,
    ROUND(l.price - m.medianPrice, 0)                  AS price_vs_median_dollars,
    ROUND((l.price - m.medianPrice) * 100.0 / m.medianPrice, 1) AS price_vs_median_pct
FROM listings AS l
INNER JOIN market AS m
    ON l.zipCode = m.zipCode
WHERE m.medianPrice IS NOT NULL
  AND m.medianPrice > 0
ORDER BY price_vs_median_pct ASC
LIMIT 15;
"""

pd.read_sql(query_4, conn)

,formattedAddress,city,zipCode,propertyType,listing_price,zip_median_price,price_vs_median_dollars,price_vs_median_pct
0,"1204 Ragan Ave, High Point, NC 27260",High Point,27260,Single Family,40000,180000,-140000.0,-77.8
1,"2893 Saint Giles Ct, High Point, NC 27262",High Point,27262,Land,70000,287500,-217500.0,-75.7
2,"1401 Pershing St, High Point, NC 27260",High Point,27260,Single Family,50000,180000,-130000.0,-72.2
3,"511 Mystic Dr, Apt E, Greensboro, NC 27406",Greensboro,27406,Condo,83000,285000,-202000.0,-70.9
4,"808 Cliffside Ave, High Point, NC 27260",High Point,27260,Single Family,55000,180000,-125000.0,-69.4
5,"517 Mystic Dr, Greensboro, NC 27406",Greensboro,27406,Condo,89900,285000,-195100.0,-68.5
6,"713 Langford Ave, High Point, NC 27260",High Point,27260,Single Family,60000,180000,-120000.0,-66.7
7,"700 Glendale Dr, Apt D, Greensboro, NC 27406",Greensboro,27406,Condo,100000,285000,-185000.0,-64.9
8,"1740 N Hamilton St, Apt H, High Point, NC 27262",High Point,27262,Condo,105000,287500,-182500.0,-63.5
9,"2326 W Vandalia Rd, Unit B, Greensboro, NC 27407",Greensboro,27407,Condo,96000,255000,-159000.0,-62.4


## Query 5 — Three-Way Join: Listings + Demographics + Market Data

**Purpose:** Build a single enriched view of each listing that combines its 
own attributes (size, beds, price), the demographic context of its zip code 
(income, renter ratio), and the market context of its zip code (median price, 
days on market, inventory). This is the foundational analytical view that 
mirrors the `df_invest` dataframe in the main notebook — but produced 
entirely in SQL.

In [64]:
query_5 = """
SELECT
    l.formattedAddress,
    l.city,
    l.zipCode,
    l.propertyType,
    l.price                                    AS listing_price,
    l.bedrooms,
    l.bathrooms,
    l.squareFootage,
    ROUND(l.price * 1.0 / l.squareFootage, 0)  AS price_per_sqft,
    c.median_household_income,
    ROUND(c.renter_ratio * 100, 1)             AS renter_pct,
    m.medianPrice                              AS zip_median_price,
    m.medianDaysOnMarket                       AS zip_median_dom,
    m.totalListings                            AS zip_total_inventory
FROM listings AS l
INNER JOIN census AS c
    ON l.zipCode = c.zipCode
INNER JOIN market AS m
    ON l.zipCode = m.zipCode
WHERE l.squareFootage IS NOT NULL
  AND l.squareFootage > 0
  AND m.medianPrice IS NOT NULL
ORDER BY l.price ASC
LIMIT 15;
"""

pd.read_sql(query_5, conn)

,formattedAddress,city,zipCode,propertyType,listing_price,bedrooms,bathrooms,squareFootage,price_per_sqft,median_household_income,renter_pct,zip_median_price,zip_median_dom,zip_total_inventory
0,"1204 Ragan Ave, High Point, NC 27260",High Point,27260,Single Family,40000,2.0,1.0,624.0,64.0,41817,51.7,180000,44,134
1,"1401 Pershing St, High Point, NC 27260",High Point,27260,Single Family,50000,2.0,1.0,768.0,65.0,41817,51.7,180000,44,134
2,"808 Cliffside Ave, High Point, NC 27260",High Point,27260,Single Family,55000,4.0,2.0,1440.0,38.0,41817,51.7,180000,44,134
3,"713 Langford Ave, High Point, NC 27260",High Point,27260,Single Family,60000,4.0,2.0,1428.0,42.0,41817,51.7,180000,44,134
4,"2893 Saint Giles Ct, High Point, NC 27262",High Point,27262,Land,70000,4.0,4.0,3260.0,21.0,64395,35.9,287500,36,148
5,"531 W Ward Ave, High Point, NC 27260",High Point,27260,Single Family,75000,2.0,1.0,784.0,96.0,41817,51.7,180000,44,134
6,"517 Flint Ave, High Point, NC 27260",High Point,27260,Single Family,80000,4.0,2.0,1344.0,60.0,41817,51.7,180000,44,134
7,"511 Mystic Dr, Apt E, Greensboro, NC 27406",Greensboro,27406,Condo,83000,2.0,1.0,715.0,116.0,56748,41.7,285000,30,245
8,"517 Mystic Dr, Greensboro, NC 27406",Greensboro,27406,Condo,89900,2.0,1.0,700.0,128.0,56748,41.7,285000,30,245
9,"2326 W Vandalia Rd, Unit B, Greensboro, NC 27407",Greensboro,27407,Condo,96000,2.0,2.0,953.0,101.0,59308,48.4,255000,42,217


## Query 6 — Rank Listings Within Each Zip by Price (Window Function)

**Purpose:** For each zip code, identify the cheapest listings ranked from 
lowest to highest price. Unlike a simple ORDER BY price, this query produces 
a *separate ranking within each zip code*, so we can answer "show me the 3 
cheapest listings in EVERY zip" — useful for an investor who wants to scan 
opportunities across multiple neighborhoods at once.

In [65]:
query_6 = """
SELECT *
FROM (
    SELECT
        zipCode,
        city,
        formattedAddress,
        propertyType,
        price,
        bedrooms,
        squareFootage,
        ROW_NUMBER() OVER (
            PARTITION BY zipCode
            ORDER BY price ASC
        ) AS price_rank_in_zip
    FROM listings
    WHERE price IS NOT NULL
) AS ranked
WHERE price_rank_in_zip <= 3
ORDER BY zipCode, price_rank_in_zip;
"""

pd.read_sql(query_6, conn)

,zipCode,city,formattedAddress,propertyType,price,bedrooms,squareFootage,price_rank_in_zip
0,27260,High Point,"1204 Ragan Ave, High Point, NC 27260",Single Family,40000,2.0,624.0,1
1,27260,High Point,"1401 Pershing St, High Point, NC 27260",Single Family,50000,2.0,768.0,2
2,27260,High Point,"808 Cliffside Ave, High Point, NC 27260",Single Family,55000,4.0,1440.0,3
3,27262,High Point,"2893 Saint Giles Ct, High Point, NC 27262",Land,70000,4.0,3260.0,1
4,27262,High Point,"1740 N Hamilton St, Apt H, High Point, NC 27262",Condo,105000,2.0,992.0,2
5,27262,High Point,"204 Northpoint Ave Unit C, High Point, NC 27262",Condo,108000,1.0,659.0,3
6,27263,High Point,"606 E Springfield Rd, High Point, NC 27263",Single Family,45000,2.0,1606.0,1
7,27263,High Point,"6111 Appletree Ct, High Point, NC 27263",Single Family,62400,3.0,1160.0,2
8,27263,High Point,"312 Model Farm Rd, High Point, NC 27263",Single Family,75000,2.0,912.0,3
9,27265,High Point,"381 James Ct, High Point, NC 27265",Townhouse,159900,2.0,942.0,1


## Query 7 — Rank Listings by Price-per-Sqft Within Each City (Window Function)

**Purpose:** For each city, identify the listings offering the most square 
footage per dollar — a more sophisticated value metric than raw price. A 
200k listing at 2,000 sqft (USD 100/sqft) is a better per-dollar value than a 
150k listing at 1,000 sqft (USD 150/sqft), even though it costs more. This 
query surfaces the top value-per-sqft listings in each city using a window 
function partitioned by city.

In [66]:
query_7 = """
SELECT *
FROM (
    SELECT
        city,
        zipCode,
        formattedAddress,
        propertyType,
        price,
        squareFootage,
        ROUND(price * 1.0 / squareFootage, 0) AS price_per_sqft,
        RANK() OVER (
            PARTITION BY city
            ORDER BY (price * 1.0 / squareFootage) ASC
        ) AS value_rank_in_city
    FROM listings
    WHERE squareFootage IS NOT NULL
      AND squareFootage > 0
      AND price IS NOT NULL
) AS ranked
WHERE value_rank_in_city <= 10
ORDER BY city, value_rank_in_city;
"""

pd.read_sql(query_7, conn)

,city,zipCode,formattedAddress,propertyType,price,squareFootage,price_per_sqft,value_rank_in_city
0,Greensboro,27406,"523 W Terrell St, Greensboro, NC 27406",Single Family,128000,1987.0,64.0,1
1,Greensboro,27401,"1101 N Elm St, Unit 802, Greensboro, NC 27401",Condo,129900,1398.0,93.0,2
2,Greensboro,27406,"409 Burtner St, Greensboro, NC 27406",Single Family,140000,1395.0,100.0,3
3,Greensboro,27406,"4220 Farlow Dr, Greensboro, NC 27406",Single Family,364900,3623.0,101.0,4
4,Greensboro,27407,"2326 W Vandalia Rd, Unit B, Greensboro, NC 27407",Condo,96000,953.0,101.0,5
5,Greensboro,27401,"1101 N Elm St, Unit 1101, Greensboro, NC 27401",Condo,139900,1336.0,105.0,6
6,Greensboro,27407,"2328 W Vandalia Rd, Unit G, Greensboro, NC 27407",Condo,102000,947.0,108.0,7
7,Greensboro,27406,"2414 Atlanta St, Greensboro, NC 27406",Single Family,214900,1972.0,109.0,8
8,Greensboro,27409,"508 Stage Coach Ct, Greensboro, NC 27409",Single Family,350000,3196.0,110.0,9
9,Greensboro,27406,"938 Martin St, Greensboro, NC 27406",Single Family,135000,1203.0,112.0,10


## Query 8 — Listings Priced Below Their Zip's Median (Correlated Subquery)

**Purpose:** Identify listings priced below the median price of their specific 
zip code, computed entirely in SQL without pre-aggregating. For each listing, 
the correlated subquery looks up the median price for that listing's zip code 
and compares it to the listing's own price. This is the SQL implementation of 
the `price_vs_zip_median_pct` feature engineered in the main notebook — but 
expressed declaratively in one statement.

In [68]:
query_8 = """
SELECT
    l.formattedAddress,
    l.city,
    l.zipCode,
    l.propertyType,
    l.price                                AS listing_price,
    ROUND(
        (SELECT AVG(price)
         FROM listings AS sub
         WHERE sub.zipCode = l.zipCode
           AND sub.price IS NOT NULL),
        0
    )                                      AS zip_avg_price,
    ROUND(
        l.price - (SELECT AVG(price)
                   FROM listings AS sub
                   WHERE sub.zipCode = l.zipCode
                     AND sub.price IS NOT NULL),
        0
    )                                      AS price_vs_zip_avg,
    ROUND(
        (l.price - (SELECT AVG(price)
                    FROM listings AS sub
                    WHERE sub.zipCode = l.zipCode
                      AND sub.price IS NOT NULL))
        * 100.0
        / (SELECT AVG(price)
           FROM listings AS sub
           WHERE sub.zipCode = l.zipCode
             AND sub.price IS NOT NULL),
        1
    )                                      AS price_vs_zip_avg_pct
FROM listings AS l
WHERE l.price IS NOT NULL
ORDER BY price_vs_zip_avg_pct ASC
LIMIT 15;
"""

pd.read_sql(query_8, conn)

,formattedAddress,city,zipCode,propertyType,listing_price,zip_avg_price,price_vs_zip_avg,price_vs_zip_avg_pct
0,"606 E Springfield Rd, High Point, NC 27263",High Point,27263,Single Family,45000,288520.0,-243520.0,-84.4
1,"1204 Ragan Ave, High Point, NC 27260",High Point,27260,Single Family,40000,197228.0,-157228.0,-79.7
2,"6111 Appletree Ct, High Point, NC 27263",High Point,27263,Single Family,62400,288520.0,-226120.0,-78.4
3,"2893 Saint Giles Ct, High Point, NC 27262",High Point,27262,Land,70000,311384.0,-241384.0,-77.5
4,"1401 Pershing St, High Point, NC 27260",High Point,27260,Single Family,50000,197228.0,-147228.0,-74.6
5,"2326 W Vandalia Rd, Unit B, Greensboro, NC 27407",Greensboro,27407,Condo,96000,376851.0,-280851.0,-74.5
6,"312 Model Farm Rd, High Point, NC 27263",High Point,27263,Single Family,75000,288520.0,-213520.0,-74.0
7,"2328 W Vandalia Rd, Unit G, Greensboro, NC 27407",Greensboro,27407,Condo,102000,376851.0,-274851.0,-72.9
8,"511 Mystic Dr, Apt E, Greensboro, NC 27406",Greensboro,27406,Condo,83000,303940.0,-220940.0,-72.7
9,"808 Cliffside Ave, High Point, NC 27260",High Point,27260,Single Family,55000,197228.0,-142228.0,-72.1


## Query 9 — Top Rental-Demand Zips and Their Inventory (Chained CTEs)

**Purpose:** Identify the top 5 zip codes by renter ratio, then surface the 
4 cheapest sub-$250k listings within each. The chained CTE pattern computes 
the strategic filter (highest-renter zips) first, then the tactical view 
(per-zip price ranking) second, then assembles the final result. This 
mirrors how a real investor would think — narrow by neighborhood quality, 
then by inventory at acceptable price points.

In [70]:
query_9 = """
WITH top_renter_zips AS (
    SELECT
        zipCode,
        renter_ratio,
        median_household_income,
        total_population
    FROM census
    WHERE renter_ratio IS NOT NULL
    ORDER BY renter_ratio DESC
    LIMIT 5
),
ranked_inventory AS (
    SELECT
        t.zipCode,
        t.renter_ratio,
        t.median_household_income,
        l.formattedAddress,
        l.city,
        l.propertyType,
        l.price,
        l.bedrooms,
        l.squareFootage,
        ROW_NUMBER() OVER (
            PARTITION BY t.zipCode
            ORDER BY l.price ASC
        ) AS price_rank_in_zip
    FROM top_renter_zips AS t
    INNER JOIN listings AS l
        ON l.zipCode = t.zipCode
    WHERE l.price <= 250000
      AND l.squareFootage IS NOT NULL
      AND l.squareFootage > 0
)
SELECT
    zipCode,
    ROUND(renter_ratio * 100, 1)              AS renter_pct,
    median_household_income,
    formattedAddress,
    city,
    propertyType,
    price,
    bedrooms,
    squareFootage,
    ROUND(price * 1.0 / squareFootage, 0)     AS price_per_sqft
FROM ranked_inventory
WHERE price_rank_in_zip <= 4
ORDER BY renter_ratio DESC, price ASC;
"""

pd.read_sql(query_9, conn)

,zipCode,renter_pct,median_household_income,formattedAddress,city,propertyType,price,bedrooms,squareFootage,price_per_sqft
0,27409,59.6,57862,"5661 Hornaday Rd, Unit H, Greensboro, NC 27409",Greensboro,Condo,114000,2.0,788.0,145.0
1,27409,59.6,57862,"5715 Bramblegate Rd, Unit F, Greensboro, NC 27409",Greensboro,Condo,129900,2.0,1152.0,113.0
2,27409,59.6,57862,"5515 Hornaday Rd, Unit D, Greensboro, NC 27409",Greensboro,Condo,133000,2.0,912.0,146.0
3,27409,59.6,57862,"5737 Bramblegate Rd, Unit F, Greensboro, NC 27409",Greensboro,Condo,139999,2.0,1070.0,131.0
4,27401,54.2,44524,"1101 N Elm St, Unit 802, Greensboro, NC 27401",Greensboro,Condo,129900,3.0,1398.0,93.0
5,27401,54.2,44524,"1013 N Elm St, Apt B6, Greensboro, NC 27401",Greensboro,Condo,134900,1.0,621.0,217.0
6,27401,54.2,44524,"1101 N Elm St, Unit 206, Greensboro, NC 27401",Greensboro,Condo,135900,2.0,1083.0,125.0
7,27401,54.2,44524,"1101 N Elm St, Unit 104, Greensboro, NC 27401",Greensboro,Condo,139500,2.0,1056.0,132.0
8,27260,51.7,41817,"1204 Ragan Ave, High Point, NC 27260",High Point,Single Family,40000,2.0,624.0,64.0
9,27260,51.7,41817,"1401 Pershing St, High Point, NC 27260",High Point,Single Family,50000,2.0,768.0,65.0


## Query 10 — "Value Rental Territory": High Renter Demand + Below-Average Income

**Purpose:** Identify zip codes that meet two conditions simultaneously — 
above-average renter ratio (strong rental demand) AND below-average median 
income (working-class neighborhoods where entry-level rentals are most 
needed). These zips represent the highest-value rental investment territory: 
demand for rentals is strong, but property prices are still accessible 
because incomes are lower. This is a cross-cutting screen rather than a 
simple ranking.

In [71]:
query_10 = """
SELECT
    c.zipCode,
    ROUND(c.renter_ratio * 100, 1)             AS renter_pct,
    c.median_household_income,
    c.total_population,
    COUNT(l.id)                                AS listing_count,
    ROUND(AVG(l.price), 0)                     AS avg_listing_price,
    ROUND(AVG(l.price * 1.0 / l.squareFootage), 0) AS avg_price_per_sqft
FROM census AS c
LEFT JOIN listings AS l
    ON l.zipCode = c.zipCode
   AND l.squareFootage IS NOT NULL
   AND l.squareFootage > 0
WHERE c.renter_ratio > (
        SELECT AVG(renter_ratio) FROM census WHERE renter_ratio IS NOT NULL
    )
  AND c.median_household_income < (
        SELECT AVG(median_household_income)
        FROM census
        WHERE median_household_income > 0
    )
GROUP BY c.zipCode, c.renter_ratio, c.median_household_income, c.total_population
ORDER BY c.renter_ratio DESC;
"""

pd.read_sql(query_10, conn)

,zipCode,renter_pct,median_household_income,total_population,listing_count,avg_listing_price,avg_price_per_sqft
0,27409,59.6,57862,16927,36,246644.0,166.0
1,27101,56.4,44611,21881,0,NaN,NaN
2,27401,54.2,44524,21788,55,275574.0,196.0
3,27260,51.7,41817,25941,71,197228.0,157.0
4,27407,48.4,59308,52583,77,376851.0,199.0
5,27405,46.7,46498,52525,129,287404.0,175.0
6,27403,41.7,56722,20675,33,386000.0,203.0
7,27406,41.7,56748,63661,109,303940.0,166.0
8,27105,37.8,41580,41257,0,NaN,NaN


## Query 11 — Per-City Executive Summary

**Purpose:** Produce a clean one-row-per-city summary suitable for executive 
reporting. Each city is summarized on inventory count, price distribution 
(min, percentile, max), days on market, and bedroom mix. This is the kind 
of view that would lead an investor deck — "here's the high-level picture 
of each market we analyzed."

In [72]:
query_11 = """
SELECT
    city,
    COUNT(*)                                                AS total_listings,
    ROUND(AVG(price), 0)                                    AS avg_price,
    ROUND(MIN(price), 0)                                    AS min_price,
    ROUND(MAX(price), 0)                                    AS max_price,
    ROUND(AVG(squareFootage), 0)                            AS avg_sqft,
    ROUND(AVG(price * 1.0 / squareFootage), 0)              AS avg_price_per_sqft,
    ROUND(AVG(bedrooms), 1)                                 AS avg_bedrooms,
    ROUND(AVG(bathrooms), 1)                                AS avg_bathrooms,
    ROUND(AVG(yearBuilt), 0)                                AS avg_year_built,
    SUM(CASE WHEN has_hoa = 1 THEN 1 ELSE 0 END)            AS listings_with_hoa,
    ROUND(
        SUM(CASE WHEN has_hoa = 1 THEN 1.0 ELSE 0.0 END) * 100 / COUNT(*),
        1
    )                                                       AS pct_with_hoa
FROM listings
WHERE squareFootage IS NOT NULL AND squareFootage > 0
GROUP BY city
ORDER BY total_listings DESC;
"""

pd.read_sql(query_11, conn)

,city,total_listings,avg_price,min_price,max_price,avg_sqft,avg_price_per_sqft,avg_bedrooms,avg_bathrooms,avg_year_built,listings_with_hoa,pct_with_hoa
0,Greensboro,695,354965.0,83000.0,998000.0,1921.0,185.0,3.2,2.3,1989.0,389,56.0
1,High Point,257,300771.0,40000.0,999000.0,1816.0,167.0,3.1,2.1,1977.0,84,32.7


## Query 12 — Property Type Market Share Ranked Within Each City

**Purpose:** For each city, calculate the inventory share of each property 
type (Single Family, Condo, Townhouse, Land, Multi-Family) and rank them 
by share within their city. Unlike Query 2's rollup view, this answers a 
sharper analytical question: "What's the dominant product type in each 
market, and how concentrated is that dominance?" A market dominated by a 
single property type (say, 80% single-family) presents different 
opportunities than a market with balanced inventory across types.


In [73]:
query_12 = """
WITH city_type_counts AS (
    SELECT
        city,
        propertyType,
        COUNT(*)                             AS type_count,
        ROUND(AVG(price), 0)                 AS avg_price
    FROM listings
    WHERE price IS NOT NULL
    GROUP BY city, propertyType
)
SELECT
    city,
    propertyType,
    type_count,
    avg_price,
    SUM(type_count) OVER (PARTITION BY city)                AS city_total_listings,
    ROUND(
        type_count * 100.0 / SUM(type_count) OVER (PARTITION BY city),
        1
    )                                                        AS market_share_pct,
    RANK() OVER (
        PARTITION BY city
        ORDER BY type_count DESC
    )                                                        AS share_rank_in_city
FROM city_type_counts
ORDER BY city, share_rank_in_city;
"""

pd.read_sql(query_12, conn)

,city,propertyType,type_count,avg_price,city_total_listings,market_share_pct,share_rank_in_city
0,Greensboro,Single Family,463,391149.0,695,66.6,1
1,Greensboro,Townhouse,145,328017.0,695,20.9,2
2,Greensboro,Condo,84,204582.0,695,12.1,3
3,Greensboro,Manufactured,2,219900.0,695,0.3,4
4,Greensboro,Multi-Family,1,412000.0,695,0.1,5
5,High Point,Single Family,211,311526.0,257,82.1,1
6,High Point,Townhouse,31,301270.0,257,12.1,2
7,High Point,Condo,13,149608.0,257,5.1,3
8,High Point,Land,1,70000.0,257,0.4,4
9,High Point,Manufactured,1,211900.0,257,0.4,4
